In [16]:
import numpy as np
import sys
import os

# Add the library path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

from ex_fuzzy_reg import fuzzy_sets as fs
from ex_fuzzy_reg import fuzzy_variable as fv
from ex_fuzzy_reg import rules_reg as r

#### Mamdani Inference:

In [17]:
bad  = fs.TriangularFS('Bad', [0, 0, 5], [0, 10])
medium = fs.TrapezoidalFS('Medium', [2.5, 4, 6, 7], [0, 10])
good = fs.TriangularFS('Good', [6, 10, 10], [0, 10])
service = fv.FuzzyVariable('Service', [bad, medium, good])

In [18]:
poor  = fs.TriangularFS('Poor', [0, 0, 4], [0, 10])
normal = fs.TrapezoidalFS('Normal', [2.5, 4, 5, 7], [0, 10])
good_food = fs.TriangularFS('Good', [6.5, 10, 10], [0, 10])
food = fv.FuzzyVariable('Food', [poor, normal, good_food])

In [19]:
low = fs.TriangularFS('Low', [0, 0, 11], [0, 20])
medium_tip = fs.TriangularFS('Medium', [10, 12, 14], [0, 20])
high = fs.TriangularFS('High', [13.5, 20, 20], [0, 20])
tip = fv.FuzzyVariable('Tip', [low, medium_tip, high])

In [ ]:
R1 = r.RuleSimple([0, 0], 0)  # Bad, Poor -> Low
R2 = r.RuleSimple([0, 1], 0)  # Bad, Normal -> Low
R3 = r.RuleSimple([1, 0], 1)  # Medium, Poor -> Medium
R4 = r.RuleSimple([2, 2], 2)  # Good, Good -> High
R5 = r.RuleSimple([1, 1], 1)  # Medium, Medium -> Medium

RB = r.RuleBaseRegT1(
    antecedents=[service, food],
    rules=[R1, R2, R3, R4, R5],
    consequent=tip
)

RB.print_rules()
print("\nRuleBase Matrix:")
print(RB.get_rulebase_matrix())

IF Service IS Bad AND Food IS Poor THEN consequent vl is 0
IF Service IS Bad AND Food IS Normal THEN consequent vl is 0
IF Service IS Medium AND Food IS Poor THEN consequent vl is 1
IF Service IS Good AND Food IS Good THEN consequent vl is 2
IF Service IS Medium AND Food IS Normal THEN consequent vl is 1


RuleBase Matrix:
[[0. 0.]
 [0. 1.]
 [1. 0.]
 [2. 2.]
 [1. 1.]] 



In [21]:
x = np.array([
    [1, 3],
    [4, 2],
    [9, 10],
    [5, 6],
    [10, 10],
    [7, 7]
])

y = RB.inference(x)
for i, (xi, yi) in enumerate(zip(x, y)):
    print(f"[{i}] Service: {xi[0]}, Food: {xi[1]} -> Tip: {yi}")

[0] Service: 1, Food: 3 -> Tip: [4.67735043]
[1] Service: 4, Food: 2 -> Tip: [8.77805279]
[2] Service: 9, Food: 10 -> Tip: [17.725]
[3] Service: 5, Food: 6 -> Tip: [12.]
[4] Service: 10, Food: 10 -> Tip: [17.83333333]
[5] Service: 7, Food: 7 -> Tip: [16.97619048]


#### TSK Inference:

In [22]:
x1_small = fs.TriangularFS('x1 small', [0, 0, 16], [0, 20])
x1_big = fs.TriangularFS('x1 big', [10, 20, 20], [0, 20])

X1 = fv.FuzzyVariable('x1', [x1_small, x1_big])

In [23]:
x2_small = fs.TriangularFS('x2 small', [0, 0, 8], [0, 10])
x2_big = fs.TriangularFS('x2 big', [2, 10, 10], [0, 10])

X2 = fv.FuzzyVariable('x2', [x2_small, x2_big])

In [24]:
R1_consq = r.ConsequentTSK([1, 1])
R2_consq = r.ConsequentTSK([2, 0])
R3_consq = r.ConsequentTSK([0, 3])

R1 = r.RuleSimpleTSK([0, 0], R1_consq)
R2 = r.RuleSimpleTSK([1, -1], R2_consq)
R3 = r.RuleSimpleTSK([-1, 1], R3_consq)

RB = r.RuleBaseRegTSK([X1, X2], [R1, R2, R3])

In [25]:
x = np.array([[12, 5]])

print(round(RB.inference(x)[0], 1))

17.8
